In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags
from src.features.rolling import add_prior_chase_rate

df = load_all_snapshots()
flagged = add_discipline_flags(df)
with_prior = add_prior_chase_rate(flagged)

print(with_prior["batter_prior_chase"].describe().round(3))
print()
early = with_prior[with_prior["game_date"] < "2024-05-01"]
late = with_prior[with_prior["game_date"] > "2024-09-01"]
print("April  std:", early["batter_prior_chase"].std().round(4))
print("Sept   std:", late["batter_prior_chase"].std().round(4))

count     710632.000
unique     30120.000
top            0.282
freq        8720.000
Name: batter_prior_chase, dtype: float64

April  std: 0.0226
Sept   std: 0.042


In [2]:
check = with_prior[with_prior["batter"] == 592450].sort_values(
    ["game_date", "at_bat_number", "pitch_number"])
first_date = check["game_date"].min()
print("first date:", first_date)
print("values on first date:", check[check["game_date"] == first_date]["batter_prior_chase"].unique())

first date: 2024-03-28
values on first date: [0.282]


In [3]:
print(with_prior["batter_prior_chase"].dtype)
print(pd.to_numeric(with_prior["batter_prior_chase"]).describe().round(4))

object
count    710632.0000
mean          0.2795
std           0.0366
min           0.1767
25%           0.2558
50%           0.2785
75%           0.3006
max           0.4313
Name: batter_prior_chase, dtype: float64


In [4]:
import numpy as np

late = with_prior[with_prior["game_date"] > "2024-09-01"].copy()
late["_oz"] = ~late["in_zone"]
oz = late[late["_oz"]]

actual = oz.groupby("batter")["is_swing"].agg(["mean", "size"])
actual = actual[actual["size"] >= 50]
prior = oz.groupby("batter")["batter_prior_chase"].first()

joined = actual.join(prior).dropna()
joined["batter_prior_chase"] = pd.to_numeric(joined["batter_prior_chase"])

print(f"{len(joined)} batters")
print("correlation between prior estimate and actual September chase:",
      round(joined["mean"].corr(joined["batter_prior_chase"]), 3))
print()
print("league mean baseline MAE:", round((joined["mean"] - 0.282).abs().mean(), 4))
print("prior estimate MAE:      ", round((joined["mean"] - joined["batter_prior_chase"]).abs().mean(), 4))

366 batters
correlation between prior estimate and actual September chase: 0.762

league mean baseline MAE: 0.0554
prior estimate MAE:       0.0377
